In [6]:
import torch
import torch.nn as nn
import numpy as np

# Preparing data

In [7]:
file = np.load('DataTrain.npy')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #moving all the weights to the GPU
data = torch.tensor(file, dtype=torch.float32)
features = data[:,:-1] #getting just the features
fraud = data[:, -1].float() #just the fraud
dataset = torch.utils.data.TensorDataset(features, fraud) #creating a dataset with separate features and fraud
loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True,num_workers=4, pin_memory=True) #separates the dataset into clusters so we can feed it just enough info

# Creatring a NN class and defining the structure 

In [8]:
class Model(nn.Module):
  def __init__(self, layerSize = [256, 128, 64, 32]):
    super(Model, self).__init__()
    layers = []
    c = features.shape[1]
    for s in layerSize:
        layers.append(nn.Linear(c, s))
        layers.append(nn.BatchNorm1d(s))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(0.2))
        c = s
    layers.append(nn.Linear(c, 1))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

# Creating an object of NN and defining loss function

In [9]:
model = Model().to(device)
print(next(model.parameters()).device)
pos_weight = (((len(fraud) - fraud.sum()) / fraud.sum())).to(device)#finding an appropriate weight cuz fraud is very rare, and it will just generate 0 all the time
optimiser = torch.optim.Adam(model.parameters(), lr=0.003) #setting learning rate and the optimisation function
loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight) #setting loss function
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser,patience=3)

cuda:0


# Training my boy 

In [10]:
end_loss = []
for epoch in range(50):
    total_loss=0
    num_batches = 0
    model.train()
    for feature,is_fraud in loader:
        feature = feature.to(device)
        is_fraud = is_fraud.to(device)
        
        optimiser.zero_grad() #reset gradient
        outputs = model(feature).squeeze() #get the predictions 
        
        angry_loss_value = loss(outputs, is_fraud) #checks how actually accurate it is 
        angry_loss_value.backward() #holy fricking legend of a back propagation

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step() #adjusts the weights so it actually learns something 
        
        #Everything after this point of code in this cell is just looking at the progress made by the model
        
        total_loss += angry_loss_value.item()
        num_batches += 1
    avg_loss = total_loss / num_batches
    scheduler.step(avg_loss)
    end_loss.append(avg_loss)
    print(f"Epoch {epoch}, Avg Loss: {avg_loss}")

Epoch 0, Avg Loss: 0.706406388359723
Epoch 1, Avg Loss: 0.6615655199134486
Epoch 2, Avg Loss: 0.6427548726968412
Epoch 3, Avg Loss: 0.6464936593940876
Epoch 4, Avg Loss: 0.6264712829106042
Epoch 5, Avg Loss: 0.6170381189122108
Epoch 6, Avg Loss: 0.5951276586147998
Epoch 7, Avg Loss: 0.5989612498939437
Epoch 8, Avg Loss: 0.5846361796163166
Epoch 9, Avg Loss: 0.5762804835370017
Epoch 10, Avg Loss: 0.567365715679471
Epoch 11, Avg Loss: 0.5633831488294325
Epoch 12, Avg Loss: 0.551536407596713
Epoch 13, Avg Loss: 0.5577584421751154
Epoch 14, Avg Loss: 0.5526866873426961
Epoch 15, Avg Loss: 0.5479903335109841
Epoch 16, Avg Loss: 0.5461350593802677
Epoch 17, Avg Loss: 0.5340295982864408
Epoch 18, Avg Loss: 0.5289471228656635
Epoch 19, Avg Loss: 0.5385140465698222
Epoch 20, Avg Loss: 0.5314265295621956
Epoch 21, Avg Loss: 0.5266192468097216
Epoch 22, Avg Loss: 0.5310890875047862
Epoch 23, Avg Loss: 0.521704490823422
Epoch 24, Avg Loss: 0.5197777410081569
Epoch 25, Avg Loss: 0.5217380523034532


In [11]:
torch.save(model.state_dict(), "model_weights3.pth")